# 출퇴근 스트레스 집계 파이프라인 (최적화 버전)

**주요 변경사항**
| 항목 | AS-IS | TO-BE |
|------|-------|-------|
| 메모리 구조 | 파일별 전처리 행 누적 → concat(100M행) | 파일별 즉시 OD 집계 → 수천 행만 보존 |
| avg_time 계산 | 파일별 산술평균 후 재평균 (오류) | popl_cnt 가중평균 (정확) |
| 정규화 시점 | 파일별 개별 적용 (스케일 불일치) | 전체 데이터 기준 1회 적용 |
| H 유형 필터 | str.contains() | isin() — 수 배 빠름 |
| copy() 호출 | Seoul 필터 내 2회 불필요 복사 | 불리언 마스크만 사용 |
| arv_dt 파싱 | 파싱 후 미사용 | USE_COLS_READ에서 제거 |
| 저장 파일 | 01_전처리후_선정데이터.csv (100M행) | 삭제 (집계 결과만 저장) |

In [ ]:
import gc
import time
import pandas as pd
import numpy as np
from pathlib import Path

# =========================================================
# 0) 설정
# =========================================================
RUN_MODE = "folder_monthly"   # "prepare_delimiter" | "single" | "folder_monthly"

INPUT_PATH = Path(r"\\98.44.8.95\janghogeun\B076. 서울시 행정동별 내국인 KT 생활이동 데이터\2. 파일데이터\2022\202201\20220101_OUTPUT.txt")
INPUT_DIR  = Path(r"\\98.44.8.95\janghogeun\B076. 서울시 행정동별 내국인 KT 생활이동 데이터\2. 파일데이터")
FILE_PATTERN = "*.txt"   # 월 폴더 내 파일 패턴 (** 불필요 — collect_input_files가 직접 탐색)

OUTPUT_DIR = Path(r"E:\project\output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TXT_SEP    = ","
TXT_ENGINE = "python"

USE_COLS_READ = [
    "start_dt", "start_emd", "arv_emd",
    "start_arv_place_type", "mvmn_time_sum", "popl_cnt",
]

CSV_DTYPES = {
    "start_emd":            str,
    "arv_emd":              str,
    "start_arv_place_type": str,
    "mvmn_time_sum":        "float32",
    "popl_cnt":             "float32",
}

PRIORITY_GROUPS = [["HW", "WH"], ["HE", "EH"], ["WE", "EW"], ["HH", "WW", "EE"]]
TARGET_RATIO    = 0.60
YEAR_FROM, YEAR_TO = 2022, 2024
FILTER_H_INCLUDED         = True
USE_SEOUL_DIRECTION_FILTER = True
SEOUL_PREFIX = "11"

FEATURE_COLS       = ["total_pop", "avg_time", "total_time"]
LITERATURE_WEIGHTS = {"pop": 0.45, "avg": 0.35, "total": 0.20}
REG_TARGET_COL     = None

H_TYPES_SET = frozenset(
    t for grp in PRIORITY_GROUPS for t in grp if "H" in t
)

print(f"H 필터 대상 유형: {H_TYPES_SET}")
print(f"분석 연도: {YEAR_FROM} ~ {YEAR_TO}")
print(f"INPUT_DIR: {INPUT_DIR}")

In [ ]:
# =========================================================
# 1) 유틸리티
# =========================================================

def minmax(s: pd.Series) -> pd.Series:
    lo, hi = s.min(), s.max()
    if pd.isna(lo) or pd.isna(hi) or hi == lo:
        return pd.Series(0.0, index=s.index)
    return (s - lo) / (hi - lo)


def normalize_weights(w: dict) -> dict:
    total = float(sum(w.values()))
    if total == 0:
        n = len(w)
        return {k: 1.0 / n for k in w}
    return {k: float(v) / total for k, v in w.items()}


def rank_spearman(df_a, df_b, key="gu", col="stress_score"):
    a = df_a[[key, col]].rename(columns={col: "a"})
    b = df_b[[key, col]].rename(columns={col: "b"})
    m = a.merge(b, on=key, how="inner")
    if len(m) < 2:
        return np.nan
    return m["a"].rank().corr(m["b"].rank(), method="pearson")


def topk_overlap(df_a, df_b, k=5, key="gu", col="stress_score"):
    if k <= 0:
        return np.nan
    sa = set(df_a.nlargest(k, col)[key])
    sb = set(df_b.nlargest(k, col)[key])
    return len(sa & sb) / k

In [ ]:
# =========================================================
# 2) I/O
# =========================================================

def replace_pipe_with_comma_in_txt(input_dir: Path, recursive: bool = False) -> pd.DataFrame:
    pattern = "**/*.txt" if recursive else "*.txt"
    logs = []
    for fp in sorted(input_dir.glob(pattern)):
        try:
            text = fp.read_text(encoding="utf-8")
        except UnicodeDecodeError:
            text = fp.read_text(encoding="cp949", errors="replace")
        changed = text.count("|")
        if changed:
            fp.write_text(text.replace("|", ","), encoding="utf-8")
        logs.append({"file": fp.name, "pipe_replaced": changed,
                     "status": "UPDATED" if changed else "SKIPPED"})
    return pd.DataFrame(logs)


def load_raw_input(path: Path) -> pd.DataFrame:
    """필요 컬럼만 지정 dtype으로 로드"""
    read_kw = dict(
        usecols=lambda c: c in set(USE_COLS_READ),
        dtype=CSV_DTYPES,
    )
    ext = path.suffix.lower()
    if ext == ".txt":
        return pd.read_csv(path, sep=TXT_SEP, engine=TXT_ENGINE, **read_kw)
    if ext == ".csv":
        return pd.read_csv(path, **read_kw)
    raise ValueError(f"Unsupported: {ext}")


def collect_input_files(input_dir: Path, file_pattern: str = "*.txt") -> list:
    """
    폴더 구조: input_dir / YYYY / YYYYMM / 파일.txt
    YEAR_FROM ~ YEAR_TO 범위의 연도 폴더를 순서대로 탐색합니다.

    ** glob 대신 직접 탐색하는 이유:
      - 네트워크 드라이브(UNC 경로)에서 ** glob이 불안정할 수 있음
      - 폴더 구조가 명확하므로 직접 순회가 더 빠르고 확실함
    """
    files = []
    for year in range(YEAR_FROM, YEAR_TO + 1):
        year_dir = input_dir / str(year)
        if not year_dir.exists():
            print(f"  [SKIP] {year}년 폴더 없음: {year_dir}")
            continue
        month_dirs = sorted(d for d in year_dir.iterdir() if d.is_dir())
        for month_dir in month_dirs:
            month_files = sorted(month_dir.glob(file_pattern))
            files.extend(month_files)
            print(f"  {year}/{month_dir.name}: {len(month_files)}개")

    if not files:
        raise FileNotFoundError(
            f"파일을 찾지 못했습니다.\n"
            f"  경로: {input_dir}\n"
            f"  연도: {YEAR_FROM}~{YEAR_TO}\n"
            f"  패턴: {file_pattern}"
        )
    print(f"\n  총 {len(files)}개 파일 수집 완료")
    return files

In [ ]:
# =========================================================
# 3) 전처리 (최적화)
# =========================================================
# 변경점
#   1. arv_dt 파싱 제거 (코드 어디서도 사용 안 함)
#   2. str.contains("H")  →  isin(H_TYPES_SET)  (수 배 빠름)
#   3. Seoul 필터 내 df.copy() 2회  →  불리언 마스크만 생성
#   4. dropna + year 필터를 앞으로 당겨 이후 연산 대상 행 수 조기 감소

def preprocess_movement(raw_df: pd.DataFrame) -> pd.DataFrame:
    df = raw_df.copy()

    df["start_dt"]             = pd.to_datetime(df["start_dt"], errors="coerce")
    df["start_emd"]            = df["start_emd"].astype(str).str.strip()
    df["arv_emd"]              = df["arv_emd"].astype(str).str.strip()
    df["start_arv_place_type"] = df["start_arv_place_type"].astype(str).str.upper().str.strip()
    df["mvmn_time_sum"]        = pd.to_numeric(df["mvmn_time_sum"], errors="coerce")
    df["popl_cnt"]             = pd.to_numeric(df["popl_cnt"],      errors="coerce")

    # 조기 결측/연도 필터 → 이후 연산 대상 행 감소
    df = df.dropna(subset=["start_dt", "mvmn_time_sum", "popl_cnt"])
    df = df[df["start_dt"].dt.year.between(YEAR_FROM, YEAR_TO)]
    if df.empty:
        return df

    df["year"]     = df["start_dt"].dt.year.astype("int16")
    df["month"]    = df["start_dt"].dt.month.astype("int8")
    df["start_gu"] = df["start_emd"].str[:5]
    df["arv_gu"]   = df["arv_emd"].str[:5]

    # isin 이 str.contains 보다 빠름
    if FILTER_H_INCLUDED:
        df = df[df["start_arv_place_type"].isin(H_TYPES_SET)]
    if df.empty:
        return df

    if USE_SEOUL_DIRECTION_FILTER:
        # copy() 없이 불리언 마스크만 생성
        start_seoul = df["start_gu"].str.startswith(SEOUL_PREFIX)
        arv_seoul   = df["arv_gu"].str.startswith(SEOUL_PREFIX)
        inbound  = df["start_arv_place_type"].isin(["HW", "HE"]) & arv_seoul
        outbound = df["start_arv_place_type"].isin(["WH", "EH"]) & start_seoul
        df = df.loc[inbound | outbound]

    return df.reset_index(drop=True)

In [ ]:
# =========================================================
# 4) OD 집계 + 이동 유형 선택
# =========================================================
# aggregate_od: 3M행 → (year,month,start_gu,arv_gu,type) 수천 행으로 즉시 압축
# total_time_w = popl_cnt × mvmn_time_sum → 이후 가중평균 복원: avg = Σw / Σpop

def aggregate_od(pre_df: pd.DataFrame) -> pd.DataFrame:
    df = pre_df.copy()
    df["total_time_w"] = df["mvmn_time_sum"] * df["popl_cnt"]
    return (
        df.groupby(
            ["year", "month", "start_gu", "arv_gu", "start_arv_place_type"],
            as_index=False, observed=True,
        )
        .agg(total_pop=("popl_cnt", "sum"), total_time_w=("total_time_w", "sum"))
    )


def select_trip_types(agg_df: pd.DataFrame, target_ratio: float = TARGET_RATIO):
    pop_by_type = agg_df.groupby("start_arv_place_type")["total_pop"].sum()
    total_pop   = float(pop_by_type.sum())
    if total_pop == 0:
        return [], 0.0
    selected, selected_pop = [], 0.0
    for grp in PRIORITY_GROUPS:
        selected.extend(grp)
        selected_pop += float(pop_by_type.reindex(grp).fillna(0).sum())
        if selected_pop / total_pop >= target_ratio:
            break
    coverage = selected_pop / total_pop
    print(f"  선택 유형: {selected}  커버리지: {coverage:.1%}")
    return selected, coverage

In [ ]:
# =========================================================
# 5) 피처 / 가중치 / 점수
# =========================================================

def make_features_from_agg(agg_df: pd.DataFrame, group_col: str = "arv_gu") -> pd.DataFrame:
    """집계 데이터로 피처 생성. avg_time = popl_cnt 가중평균 (산술평균 버그 수정)"""
    f = (
        agg_df.groupby(group_col, as_index=False, observed=True)
        .agg(total_pop=("total_pop", "sum"), total_time_w=("total_time_w", "sum"))
        .rename(columns={group_col: "gu"})
    )
    f["avg_time"]   = f["total_time_w"] / f["total_pop"]
    f["total_time"] = f["total_pop"] * f["avg_time"]  # == total_time_w
    return f[["gu", "total_pop", "avg_time", "total_time"]]


def pca_weights(feature_df: pd.DataFrame) -> dict:
    if feature_df.empty:
        return normalize_weights(LITERATURE_WEIGHTS)
    X = feature_df[FEATURE_COLS].astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    if len(X) < 2:
        return normalize_weights(LITERATURE_WEIGHTS)
    X = (X - X.mean()) / X.std(ddof=0).replace(0, np.nan)
    X = X.fillna(0.0)
    try:
        eigvals, eigvecs = np.linalg.eigh(np.cov(X.values, rowvar=False))
        load = np.abs(eigvecs[:, np.argmax(eigvals)])
        return normalize_weights({"pop": load[0], "avg": load[1], "total": load[2]})
    except Exception:
        return normalize_weights(LITERATURE_WEIGHTS)


def critic_weights(feature_df: pd.DataFrame) -> dict:
    if feature_df.empty:
        return normalize_weights(LITERATURE_WEIGHTS)
    Z = feature_df[FEATURE_COLS].astype(float)
    Z = ((Z - Z.min()) / (Z.max() - Z.min())).fillna(0.0)
    std  = Z.std(ddof=0)
    corr = Z.corr().fillna(0.0)
    cvals = {c: std[c] * (1 - corr[c]).sum() for c in FEATURE_COLS}
    return normalize_weights({"pop": cvals["total_pop"], "avg": cvals["avg_time"], "total": cvals["total_time"]})


def regression_weights(feature_df: pd.DataFrame, target_col) -> dict:
    if not target_col or target_col not in feature_df.columns:
        return None
    d = feature_df.dropna(subset=FEATURE_COLS + [target_col]).copy()
    if len(d) < 5:
        return None
    X  = d[FEATURE_COLS].astype(float)
    y  = pd.to_numeric(d[target_col], errors="coerce")
    ok = y.notna()
    X, y = X[ok], y[ok]
    if len(X) < 5:
        return None
    Xz   = ((X - X.mean()) / X.std(ddof=0).replace(0, np.nan)).fillna(0.0)
    ystd = y.std(ddof=0)
    yz   = (y - y.mean()) / (ystd if ystd else 1.0)
    beta, *_ = np.linalg.lstsq(Xz.values, yz.values, rcond=None)
    return normalize_weights({"pop": abs(beta[0]), "avg": abs(beta[1]), "total": abs(beta[2])})


def score_with_weights(feature_df: pd.DataFrame, weights: dict, model_name: str) -> pd.DataFrame:
    d = feature_df.copy()
    w = normalize_weights(weights)
    d["pop_norm"]     = minmax(d["total_pop"])
    d["avg_norm"]     = minmax(d["avg_time"])
    d["total_norm"]   = minmax(d["total_time"])
    d["stress_score"] = (
        d["pop_norm"] * w["pop"] + d["avg_norm"] * w["avg"] + d["total_norm"] * w["total"]
    ) * 100
    d["model"]   = model_name
    d["w_pop"]   = w["pop"]
    d["w_avg"]   = w["avg"]
    d["w_total"] = w["total"]
    return d.sort_values("stress_score", ascending=False).reset_index(drop=True)

In [ ]:
# =========================================================
# 6) 메인 파이프라인 (최적화)
# =========================================================
# AS-IS 병목 구조:
#   pre_parts.append(pre)          ← 100M행 누적
#   pd.concat(pre_parts)           ← 100M행 DataFrame 생성
#   run_pipeline_from_preprocessed ← 100M행 재처리
#   selected_data.to_csv(...)      ← 100M행 CSV 저장 (수십 분)
#
# TO-BE:
#   파일별 즉시 OD 집계(수천 행) → concat → 집계 데이터로만 모든 연산

def run_folder_monthly_optimized(input_dir: Path, file_pattern: str = "*.csv"):
    files = collect_input_files(input_dir, file_pattern)
    print(f"[INFO] 대상 파일: {len(files)}개")

    od_parts       = []
    file_log       = []
    total_raw_rows = 0

    for i, fp in enumerate(files, 1):
        t0 = time.time()
        print(f"[{i:>2}/{len(files)}] {fp.name}", end="  ")
        try:
            raw_df = load_raw_input(fp)
            n_raw  = len(raw_df)
            total_raw_rows += n_raw

            pre = preprocess_movement(raw_df)
            del raw_df          # 즉시 해제

            if pre.empty:
                file_log.append({"file": fp.name, "raw_rows": n_raw,
                                 "filtered_rows": 0, "status": "EMPTY"})
                print(f"EMPTY  ({time.time()-t0:.1f}s)")
                continue

            n_filtered = len(pre)
            od_parts.append(aggregate_od(pre))  # 수천 행으로 압축
            del pre
            gc.collect()

            print(f"OK  raw={n_raw:,}  filtered={n_filtered:,}  ({time.time()-t0:.1f}s)")
            file_log.append({"file": fp.name, "raw_rows": n_raw,
                             "filtered_rows": n_filtered, "status": "OK"})

        except Exception as e:
            print(f"ERROR: {e}")
            file_log.append({"file": fp.name, "raw_rows": np.nan,
                             "filtered_rows": np.nan, "status": f"ERROR: {e}"})

    file_log_df = pd.DataFrame(file_log)
    _empty = pd.DataFrame()

    if not od_parts:
        print("[WARN] 유효 데이터 없음")
        empty_out = {k: _empty for k in
                     ["meta", "all_scores", "sensitivity", "yearly_stability", "yearly_lit_scores"]}
        return empty_out, _empty, _empty, _empty, file_log_df

    # ── 전체 OD 결합 ─────────────────────────────────────────────────────────
    print("\n[결합] 파일별 집계 통합 중...")
    all_od = pd.concat(od_parts, ignore_index=True)
    del od_parts
    gc.collect()

    # 동일 키가 여러 파일에 걸칠 경우 재집계
    all_od = (
        all_od.groupby(
            ["year", "month", "start_gu", "arv_gu", "start_arv_place_type"],
            as_index=False, observed=True,
        )
        .agg(total_pop=("total_pop", "sum"), total_time_w=("total_time_w", "sum"))
    )

    # ── 이동 유형 선택 ────────────────────────────────────────────────────────
    print("\n[이동 유형 선택]")
    selected_types, coverage = select_trip_types(all_od, TARGET_RATIO)
    od_sel = all_od[all_od["start_arv_place_type"].isin(selected_types)].copy()
    del all_od

    # ── 전체 기간 피처 / 모델별 점수 ─────────────────────────────────────────
    print("\n[모델 점수 산출]")
    f_overall = make_features_from_agg(od_sel, "arv_gu")
    w_lit     = normalize_weights(LITERATURE_WEIGHTS)
    w_pca     = pca_weights(f_overall)
    w_critic  = critic_weights(f_overall)
    w_reg     = regression_weights(f_overall, REG_TARGET_COL)

    frames = [
        score_with_weights(f_overall, w_lit,    "literature"),
        score_with_weights(f_overall, w_pca,    "pca"),
        score_with_weights(f_overall, w_critic, "critic"),
    ]
    if w_reg:
        frames.append(score_with_weights(f_overall, w_reg, "regression"))
    all_scores = pd.concat(frames, ignore_index=True)

    base_lit = all_scores[all_scores["model"] == "literature"].copy()
    sensitivity = pd.DataFrame([
        {
            "model":               m,
            "spearman_vs_lit":     rank_spearman(base_lit, all_scores[all_scores["model"] == m].copy()),
            "top5_overlap_vs_lit": topk_overlap(base_lit,  all_scores[all_scores["model"] == m].copy(), k=5),
        }
        for m in all_scores["model"].unique()
    ])

    # ── 연도별 점수 / 안정성 ──────────────────────────────────────────────────
    yearly_od = (
        od_sel.groupby(["year", "arv_gu"], as_index=False, observed=True)
        .agg(total_pop=("total_pop", "sum"), total_time_w=("total_time_w", "sum"))
    )
    yearly_od["avg_time"]   = yearly_od["total_time_w"] / yearly_od["total_pop"]
    yearly_od["total_time"] = yearly_od["total_pop"] * yearly_od["avg_time"]

    yearly_lit_parts = []
    for y in sorted(yearly_od["year"].unique()):
        ydf = (
            yearly_od[yearly_od["year"] == y]
            [["arv_gu", "total_pop", "avg_time", "total_time"]]
            .rename(columns={"arv_gu": "gu"})
            .copy()
        )
        sy = score_with_weights(ydf, w_lit, f"literature_{y}")
        sy["year"] = y
        yearly_lit_parts.append(sy[["year", "gu", "stress_score"]])
    yearly_lit_scores = (
        pd.concat(yearly_lit_parts, ignore_index=True) if yearly_lit_parts else _empty
    )

    years = sorted(yearly_lit_scores["year"].unique()) if len(yearly_lit_scores) else []
    stability_rows = []
    for i in range(len(years)):
        for j in range(i + 1, len(years)):
            y1, y2 = years[i], years[j]
            a = yearly_lit_scores[yearly_lit_scores["year"] == y1]
            b = yearly_lit_scores[yearly_lit_scores["year"] == y2]
            stability_rows.append({"year_pair": f"{y1}-{y2}",
                                   "spearman": rank_spearman(a, b)})
    yearly_stability = pd.DataFrame(stability_rows)

    # ── 월별 × 자치구 스트레스 점수 (메인 출력) ───────────────────────────────
    # 정규화를 전체 데이터 기준으로 1회만 적용
    # built-in transform("min"/"max") 이 custom function보다 빠름
    print("\n[월별 스트레스 점수 산출]")
    monthly_gu = (
        od_sel.groupby(["year", "month", "arv_gu"], as_index=False, observed=True)
        .agg(total_pop=("total_pop", "sum"), total_time_w=("total_time_w", "sum"))
    )
    monthly_gu["avg_time"]   = monthly_gu["total_time_w"] / monthly_gu["total_pop"]
    monthly_gu["total_time"] = monthly_gu["total_pop"] * monthly_gu["avg_time"]

    for feat, norm_col in [
        ("total_pop",  "pop_norm"),
        ("avg_time",   "avg_norm"),
        ("total_time", "total_norm"),
    ]:
        grp = monthly_gu.groupby(["year", "month"])[feat]
        lo  = grp.transform("min")
        hi  = grp.transform("max")
        rng = (hi - lo).replace(0, np.nan)
        monthly_gu[norm_col] = ((monthly_gu[feat] - lo) / rng).fillna(0.0)

    monthly_gu["stress_score"] = (
        monthly_gu["pop_norm"]   * w_lit["pop"]
        + monthly_gu["avg_norm"]   * w_lit["avg"]
        + monthly_gu["total_norm"] * w_lit["total"]
    ) * 100

    monthly_commute_score_df = (
        monthly_gu[["year", "month", "arv_gu", "stress_score"]]
        .rename(columns={"arv_gu": "target_gu"})
        .sort_values(["year", "month", "target_gu"])
        .reset_index(drop=True)
    )

    # ── OD 월별 집계 / 요약 ───────────────────────────────────────────────────
    monthly_agg_df = od_sel.copy()
    monthly_agg_df["yyyymm"] = (
        monthly_agg_df["year"].astype(str) + "-"
        + monthly_agg_df["month"].astype(str).str.zfill(2)
    )
    monthly_agg_df["avg_travel_time"] = (
        monthly_agg_df["total_time_w"] / monthly_agg_df["total_pop"]
    )

    monthly_summary_df = (
        od_sel.groupby(["year", "month"], as_index=False)
        .agg(total_pop=("total_pop", "sum"))
        .assign(yyyymm=lambda d:
            d["year"].astype(str) + "-" + d["month"].astype(str).str.zfill(2)
        )
        .sort_values("yyyymm")
    )

    meta = pd.DataFrame([{
        "year_range":     f"{YEAR_FROM}~{YEAR_TO}",
        "h_filter":       FILTER_H_INCLUDED,
        "selected_types": ", ".join(selected_types),
        "coverage":       round(coverage, 4),
        "raw_rows":       total_raw_rows,
        "selected_pop":   float(od_sel["total_pop"].sum()),
    }])

    combined_outputs = {
        "meta":              meta,
        "all_scores":        all_scores,
        "sensitivity":       sensitivity,
        "yearly_stability":  yearly_stability,
        "yearly_lit_scores": yearly_lit_scores,
    }

    print("\n[완료]")
    return combined_outputs, monthly_agg_df, monthly_summary_df, monthly_commute_score_df, file_log_df

In [ ]:
# =========================================================
# 7) single 모드용 래퍼 (기존 호환)
# =========================================================

def run_pipeline_single(raw_df: pd.DataFrame) -> dict:
    pre              = preprocess_movement(raw_df)
    od               = aggregate_od(pre)
    types, coverage  = select_trip_types(od)
    od_sel           = od[od["start_arv_place_type"].isin(types)].copy()
    f                = make_features_from_agg(od_sel)
    w_lit            = normalize_weights(LITERATURE_WEIGHTS)
    w_pca            = pca_weights(f)
    w_critic         = critic_weights(f)
    w_reg            = regression_weights(f, REG_TARGET_COL)

    frames = [
        score_with_weights(f, w_lit,    "literature"),
        score_with_weights(f, w_pca,    "pca"),
        score_with_weights(f, w_critic, "critic"),
    ]
    if w_reg:
        frames.append(score_with_weights(f, w_reg, "regression"))
    all_scores = pd.concat(frames, ignore_index=True)

    return {
        "meta":        pd.DataFrame([{"coverage": coverage, "raw_rows": len(raw_df),
                                       "selected_types": ", ".join(types)}]),
        "all_scores":  all_scores,
        "sensitivity": pd.DataFrame(),
        "yearly_stability":  pd.DataFrame(),
        "yearly_lit_scores": pd.DataFrame(),
    }

In [ ]:
# =========================================================
# 8) 실행
# =========================================================

def save_append(df: pd.DataFrame, path: Path):
    """파일이 이미 있으면 이어쓰기, 없으면 새로 생성"""
    if path.exists() and path.stat().st_size > 0:
        df.to_csv(path, mode="a", header=False, index=False, encoding="utf-8-sig")
        kb = path.stat().st_size // 1024
        print(f"  [추가] {path.name:<35} +{len(df):,}행  (누적 {kb:,} KB)")
    else:
        df.to_csv(path, index=False, encoding="utf-8-sig")
        kb = path.stat().st_size // 1024
        print(f"  [신규] {path.name:<35}  {len(df):,}행  ({kb:,} KB)")


if RUN_MODE == "prepare_delimiter":
    log = replace_pipe_with_comma_in_txt(INPUT_DIR)
    log.to_csv(OUTPUT_DIR / "00_delimiter_replace_log.csv", index=False, encoding="utf-8-sig")
    print("[DONE] 구분자 변환 완료")

elif RUN_MODE == "single":
    raw_df  = load_raw_input(INPUT_PATH)
    outputs = run_pipeline_single(raw_df)
    outputs["meta"].to_csv(      OUTPUT_DIR / "00_메타정보.csv",       index=False, encoding="utf-8-sig")
    outputs["all_scores"].to_csv(OUTPUT_DIR / "02_모형별_점수결과.csv", index=False, encoding="utf-8-sig")
    print("[DONE] single 모드 완료")

elif RUN_MODE == "folder_monthly":
    (
        outputs,
        monthly_agg_df,
        monthly_summary_df,
        monthly_commute_score_df,
        file_log_df,
    ) = run_folder_monthly_optimized(INPUT_DIR, FILE_PATTERN)

    print("\n[저장]")
    # 덮어쓰기 파일 (실행마다 갱신)
    outputs["meta"].to_csv(              OUTPUT_DIR / "00_메타정보.csv",           index=False, encoding="utf-8-sig")
    outputs["all_scores"].to_csv(        OUTPUT_DIR / "02_모형별_점수결과.csv",     index=False, encoding="utf-8-sig")
    outputs["sensitivity"].to_csv(       OUTPUT_DIR / "03_민감도분석.csv",          index=False, encoding="utf-8-sig")
    outputs["yearly_stability"].to_csv(  OUTPUT_DIR / "04_연도안정성분석.csv",      index=False, encoding="utf-8-sig")
    outputs["yearly_lit_scores"].to_csv( OUTPUT_DIR / "05_연도별점수_문헌기반.csv", index=False, encoding="utf-8-sig")

    # 이어쓰기 파일 (실행마다 누적)
    save_append(monthly_commute_score_df, OUTPUT_DIR / "월별_통근스트레스_점수.csv")
    save_append(monthly_agg_df,           OUTPUT_DIR / "월별_통합집계.csv")
    save_append(monthly_summary_df,       OUTPUT_DIR / "월별_요약.csv")
    save_append(file_log_df,              OUTPUT_DIR / "월별_파일처리로그.csv")

    print(f"\n[완료] → {OUTPUT_DIR}")

else:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")